# MPECSS - MacMPEC Seed Robustness (Seed 123)

This notebook runs a single-seed robustness experiment on the 191 MacMPEC problems.

- Timeout per problem: 1800 s
- Seed: 123
- Output root: /kaggle/working/outputs_macmpec_seed_123

Run the setup cell first. After a Kaggle restart, re-run setup, then use Resume instead of Fresh Run.


In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys

DATASET = 'macmpec'
RUN_TAG_PREFIX = 'Kaggle_MacMPEC_SeedRobustness_Seed123'
WORKERS = 4
TIMEOUT = 1800
NUM_PROBLEM = None  # None = all problems; set 5/10/etc for quick tests
DEFAULT_SEED = 123
SAVE_LOGS = True
SHUFFLE = True

REPO_DIR = '/kaggle/working/mpecss-kaggle'
REPO_GIT_URL = 'https://github.com/mpecssalgorithm/mpecss.git'
OUTPUT_ROOT = '/kaggle/working/outputs_macmpec_seed_123'
DATASET_PATH = os.path.join(REPO_DIR, 'benchmarks', 'macmpec', 'macmpec-json')
REPO_COMMIT = '6117ae6aa2e118936ca2ada4c44b175d091ce8ad'

RUN_PLAN = [
    {
        "slug": "seed-123",
        "tag": "Kaggle_MacMPEC_SeedRobustness_Seed123",
        "seed": 123
    }
]

repo_path = Path(REPO_DIR)
if repo_path.exists():
    shutil.rmtree(repo_path)


print(f'Cloning: {REPO_GIT_URL} @ {REPO_COMMIT}')
subprocess.run(['git', 'clone', '--depth', '1', REPO_GIT_URL, str(repo_path)], check=True)
subprocess.run(['git', 'checkout', REPO_COMMIT], check=True, cwd=str(repo_path))
sys.path.insert(0, str(repo_path))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_path)], check=True)

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
EXPECTED_PROBLEM_COUNT = 191


def _count_json_files(path):
    path = Path(path)
    if not path.is_dir():
        return 0
    return sum(1 for item in path.iterdir() if item.is_file() and item.name.endswith('.json'))


def _resolve_benchmark_dataset_path():
    checked = []
    seen = set()
    json_dir_name = Path(DATASET_PATH).name
    candidates = [Path(DATASET_PATH)]

    if KAGGLE_INPUT_ROOT.exists():
        patterns = [
            f'benchmarks/{DATASET}/{json_dir_name}',
            f'*/benchmarks/{DATASET}/{json_dir_name}',
            f'*/{DATASET}/{json_dir_name}',
            json_dir_name,
            f'*/{json_dir_name}',
            f'*/{DATASET}',
            f'**/{json_dir_name}',
        ]
        for pattern in patterns:
            candidates.extend(sorted(KAGGLE_INPUT_ROOT.glob(pattern)))

    for candidate in candidates:
        possible_paths = [candidate]
        if candidate.is_dir() and candidate.name != json_dir_name:
            possible_paths.append(candidate / json_dir_name)

        for possible in possible_paths:
            possible = possible.resolve()
            if possible in seen:
                continue
            seen.add(possible)
            checked.append(str(possible))
            count = _count_json_files(possible)
            if count > 0:
                return str(possible), count

    raise RuntimeError(
        f'{DATASET} JSON dataset not found. Add a Kaggle Input dataset containing '
        f'benchmarks/{DATASET}/{json_dir_name}/*.json or a {json_dir_name} folder. Checked:\n  '
        + '\n  '.join(checked)
    )


DATASET_PATH, problem_count = _resolve_benchmark_dataset_path()

print(f'[OK] Repo ready: {repo_path}')
print(f'[OK] Benchmark path: {DATASET_PATH}')
if problem_count != EXPECTED_PROBLEM_COUNT:
    print(f'[WARN] Expected {EXPECTED_PROBLEM_COUNT} {DATASET} problems, found {problem_count}')
print(f'[OK] {DATASET} problems found: {problem_count}')

subprocess.run([sys.executable, os.path.join(str(repo_path), 'mpecss', 'helpers', 'preflight_checks.py')], check=True)


In [ ]:
from kaggle_setup.study_runner import run_study_plan


def run_fresh():
    run_study_plan(
        RUN_PLAN,
        repo_dir=REPO_DIR,
        dataset=DATASET,
        path=DATASET_PATH,
        output_root=OUTPUT_ROOT,
        workers=WORKERS,
        timeout=TIMEOUT,
        seed=DEFAULT_SEED,
        save_logs=SAVE_LOGS,
        shuffle=SHUFFLE,
        num_problems=NUM_PROBLEM,
    )


def run_resume(retry_failed=False):
    run_study_plan(
        RUN_PLAN,
        repo_dir=REPO_DIR,
        dataset=DATASET,
        path=DATASET_PATH,
        output_root=OUTPUT_ROOT,
        workers=WORKERS,
        timeout=TIMEOUT,
        seed=DEFAULT_SEED,
        save_logs=SAVE_LOGS,
        shuffle=SHUFFLE,
        num_problems=NUM_PROBLEM,
        resume_latest=True,
        retry_failed=retry_failed,
    )


def run_summary():
    run_study_plan(
        RUN_PLAN,
        repo_dir=REPO_DIR,
        dataset=DATASET,
        path=DATASET_PATH,
        output_root=OUTPUT_ROOT,
        workers=WORKERS,
        timeout=TIMEOUT,
        seed=DEFAULT_SEED,
        save_logs=SAVE_LOGS,
        shuffle=SHUFFLE,
        num_problems=NUM_PROBLEM,
        summary_only=True,
    )


In [ ]:
run_fresh()


In [ ]:
# run_resume()


## Retry Failed Only (Manual)


In [ ]:
# run_resume(retry_failed=True)


In [ ]:
run_summary()


In [ ]:
from pathlib import Path
import shutil
import os

output_root = Path(OUTPUT_ROOT)
if not output_root.exists():
    print(f"[WARN] Output root not found: {output_root}")
else:
    zip_name = f"{globals().get('RUN_TAG', globals().get('RUN_TAG_PREFIX', 'results'))}_results"
    temp_dir = output_root.parent / zip_name
    os.rename(output_root, temp_dir)
    archive_path = shutil.make_archive(str(temp_dir), "zip", root_dir=str(temp_dir.parent), base_dir=zip_name)
    os.rename(temp_dir, output_root)
    print(f"[OK] Results directory: {output_root}")
    print(f"[OK] Download archive: {archive_path}")
    for path in sorted(output_root.iterdir()):
        print(path)


In [ ]:
import casadi
import matplotlib
import numpy
import pandas
import platform
import psutil
import scipy

print('Software Versions')
print('=' * 40)
print(f'Python: {platform.python_version()}')
print(f'NumPy: {numpy.__version__}')
print(f'SciPy: {scipy.__version__}')
print(f'Pandas: {pandas.__version__}')
print(f'Matplotlib: {matplotlib.__version__}')
print(f'CasADi: {casadi.__version__}')
print(f'psutil: {psutil.__version__}')
print('=' * 40)
